In [ ]:
def analyze_video_with_model(model, video_path, output_video_path='/content/output_video.mp4', conf_threshold=0.65):

    if model is None:
        print("Модель не загружена!")
        return

    print(f"\n{'='*80}")
    print("АНАЛИЗ ВИДЕО С РАЗМЕТКОЙ ОБЪЕКТОВ")
    print(f"{'='*80}")
    print(f"Минимальная уверенность для отображения: {conf_threshold}")

    if not os.path.exists(video_path):
        print(f"Видео файл не найден: {video_path}")
        print("Загрузите видео файл в Colab или укажите правильный путь")
        return

    print(f"Входное видео: {video_path}")
    print(f"Выходное видео: {output_video_path}")

    try:
        print("Обрабатываем видео...")

        results = model.predict(
            source=video_path,
            conf=conf_threshold,
            iou=0.4,
            device='cpu',
            save=True,
            project='/content/video_results',
            name='detection',
            exist_ok=True,
            verbose=False
        )

        result_video_path = '/content/video_results/detection'
        if os.path.exists(result_video_path):
            video_files = [f for f in os.listdir(result_video_path) if f.endswith('.mp4')]
            if video_files:
                detected_video = os.path.join(result_video_path, video_files[0])
                shutil.copy(detected_video, output_video_path)
                print(f"Видео с детекцией сохранено: {output_video_path}")
                print(f"Объекты отображаются только при уверенности ≥ {conf_threshold}")
            else:
                print("Не найден обработанный видео файл")
        else:
            print("Папка с результатами не создана")

    except Exception as e:
        print(f"Ошибка при обработке видео: {e}")

def upload_and_analyze_video_high_confidence(model):

    if model is None:
        print("Модель не загружена!")
        return

    print("\nЗАГРУЗКА ВИДЕО ДЛЯ АНАЛИЗА")
    print("="*50)

    from google.colab import files

    try:
        uploaded = files.upload()

        if not uploaded:
            print("Файл не был загружен!")
            return

        video_filename = list(uploaded.keys())[0]
        video_path = f"/content/{video_filename}"

        print(f"Видео загружено: {video_filename}")
        print(f"Размер файла: {len(uploaded[video_filename]) / 1024 / 1024:.2f} MB")

        name, ext = os.path.splitext(video_filename)
        output_path = f"/content/{name}_detected_high_conf{ext}"

        return analyze_video_with_model(model, video_path, output_path, conf_threshold=0.65)

    except Exception as e:
        print(f"Ошибка при загрузке видео: {e}")
        return None

upload_and_analyze_video_high_confidence(model)